# Phase 3 - Cleaning and validation

This notebook runs the reusable cleaning and validation modules and presents their audit evidence.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
subprocess.run([sys.executable, '-m', 'src.clean'], cwd=project_root, check=True, stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, '-m', 'src.validate'], cwd=project_root, check=True, stdout=subprocess.DEVNULL)
audit = json.loads((project_root / 'reports/generated/cleaning_audit.json').read_text(encoding='utf-8'))
validation = json.loads((project_root / 'reports/generated/processed_validation.json').read_text(encoding='utf-8'))

In [2]:
{key: audit[key] for key in ['input_rows', 'output_rows', 'removed_rows', 'removal_counts_by_reason']}

{'input_rows': 15956,
 'output_rows': 15956,
 'removed_rows': 0,
 'removal_counts_by_reason': {'missing_required': 0,
  'duplicate_tile_period': 0,
  'invalid_coordinates': 0,
  'centroid_coordinate_mismatch': 0,
  'invalid_counts': 0,
  'negative_performance': 0,
  'invalid_period': 0,
  'invalid_region': 0}}

In [3]:
{
    'loaded_download_nulls': audit['null_counts_after']['avg_lat_down_ms'],
    'loaded_upload_nulls': audit['null_counts_after']['avg_lat_up_ms'],
    'complete_loaded_latency_rows': audit['loaded_latency_complete_rows'],
    'volume_counts': audit['measurement_volume_counts'],
}

{'loaded_download_nulls': 370,
 'loaded_upload_nulls': 184,
 'complete_loaded_latency_rows': 15483,
 'volume_counts': {'high': 427, 'low': 12797, 'medium': 2732}}

In [4]:
audit['sample_conversions']

[{'quadkey': '1231331013033233',
  'avg_d_kbps': 5086,
  'avg_d_mbps': 5.086,
  'avg_u_kbps': 24564,
  'avg_u_mbps': 24.564},
 {'quadkey': '1231331013211011',
  'avg_d_kbps': 11751,
  'avg_d_mbps': 11.751,
  'avg_u_kbps': 24777,
  'avg_u_mbps': 24.777},
 {'quadkey': '1231331013212303',
  'avg_d_kbps': 9207,
  'avg_d_mbps': 9.207,
  'avg_u_kbps': 2641,
  'avg_u_mbps': 2.641},
 {'quadkey': '1231331013213132',
  'avg_d_kbps': 49008,
  'avg_d_mbps': 49.008,
  'avg_u_kbps': 16555,
  'avg_u_mbps': 16.555},
 {'quadkey': '1231331013230010',
  'avg_d_kbps': 21726,
  'avg_d_mbps': 21.726,
  'avg_u_kbps': 13664,
  'avg_u_mbps': 13.664}]

In [5]:
validation

{'row_count': 15956,
 'checks': {'nonempty': True,
  'required_columns_present': True,
  'crs_is_epsg_4326': True,
  'tile_period_key_unique': True,
  'required_values_nonnull': True,
  'counts_consistent': True,
  'download_conversion_exact': True,
  'upload_conversion_exact': True,
  'measurement_volume_valid': True,
  'geometry_complete': True,
  'all_eight_adm1_present': True,
  'measurement_volume_matches_tests': True,
  'loaded_latency_flag_correct': True},
 'passed': True,
 'validated_at_utc': '2026-10-19T10:08:47.899812+00:00'}

## Interpretation

No real row was excluded in this run. Loaded-latency nulls and one zero idle-latency value remain visible because missingness and unusual values must not be silently rewritten.